In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv("./Churn.csv")

In [10]:
df = df.drop(["CustomerId", "RowNumber", "Surname"], axis=1)

In [11]:
df = pd.get_dummies(df, columns=["Geography"], dtype=int)

In [12]:
df["is_male"] = (df["Gender"]=="Male").astype(int)
df= df.drop("Gender", axis=1)

In [13]:
y = df["Exited"]
X = df[[x for x in df.columns if x != "Exited"]]

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




In [45]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [46]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [47]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                      │ (None, 64)                  │             832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 8)                   │             264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,185 (12.44 KB)

 Trainable params: 3,185 (12.44 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [49]:
tenserflow_callback = TensorBoard(log_dir="logs", histogram_freq=1)

In [54]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

In [53]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, callbacks=[tenserflow_callback,early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8529 - loss: 0.3587 - val_accuracy: 0.8595 - val_loss: 0.3398
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8586 - loss: 0.3505 - val_accuracy: 0.8600 - val_loss: 0.3412
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8565 - loss: 0.3537 - val_accuracy: 0.8635 - val_loss: 0.3406
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8575 - loss: 0.3490 - val_accuracy: 0.8625 - val_loss: 0.3397
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8595 - loss: 0.3442 - val_accuracy: 0.8615 - val_loss: 0.3411
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8572 - loss: 0.3480 - val_accuracy: 0.8580 - val_loss: 0.3426
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8576 - loss: 0.3470 - val_accuracy: 0.8570 - val_loss: 0.3414
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8565 - loss: 0.3486 - val_accu

In [56]:
model.save("models/churn.keras")

In [57]:
%load_ext tensorboard
%tensorboard --logdir logs

In [60]:
from tensorflow.keras.models import load_model

model = load_model("models/Churn.keras")
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Loss:", test_loss)
print("Accuracy:", test_acc)

C:\Users\HP\Desktop\codes\ml\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop_2', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8615 - loss: 0.3346
Loss: 0.3346444368362427
Accuracy: 0.8615000247955322


In [61]:
import pickle

with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [62]:
import pickle

with open("models/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

In [63]:
new_data = [[
    600, 40, 5, 50000, 2, 1, 1, 60000, 1, 0, 0, 1
]]

new_data = scaler.transform(new_data)
prediction = model.predict(new_data)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
[[0.03036577]]


C:\Users\HP\Desktop\codes\ml\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
